In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

from src.config import path_nested_folders, path_files
from src.utils import load_json_content

In [2]:
EVALUATION_METRICS = ["accuracy","global_precision","global_recall","global_f1score"]
CLASSIFIERS = ["MLPClassifier","RandomForest"]
Z_VALUES = [1000,5000,10000,50000,100000,500000]

#### Plot accuracy vs z value for each classifier
The main purpose of this section is to visualize how evaluation metrics retrieved for both classifiers evolve over the range of z values.<br>
Just to recall, the z values chosen for generating windowed feature datasets were 1000,5000,10000,50000,100000,500000.

In [3]:
def plot_metrics_vs_z(mapping_z_metric):
    metric = np.unique(mapping_z_metric["metric"])[0]

    os.makedirs(path_nested_folders["METRIC_EVAL_FIG"],exist_ok=True)

    fig, ax1 = plt.subplots(figsize=(12, 7))

    mapping_z_metric_MLP = mapping_z_metric[mapping_z_metric["model_name"] == "MLPClassifier"]
    mapping_z_metric_RF = mapping_z_metric[mapping_z_metric["model_name"] == "RandomForest"]

    mapping_z_metric_MLP = mapping_z_metric_MLP.sort_values(by=["z_value"])
    mapping_z_metric_RF = mapping_z_metric_RF.sort_values(by=["z_value"])

    ax1.set_title("{} vs. Time-Window Size z\n".format(metric))
    ax1.plot(mapping_z_metric_MLP["z_value"], mapping_z_metric_MLP["value"], "s-", color="tab:green", label="MLPClassifier")
    ax1.plot(mapping_z_metric_RF["z_value"], mapping_z_metric_RF["value"], "s-", color="tab:red", label="RandomForest")
    ax1.set_xlabel("Window size z")
    ax1.legend()
    ax1.grid(True)
    plt.xticks(Z_VALUES)
    plt.yticks(np.arange(0,1.4,0.2))
    fig.tight_layout()
    
    filename = os.path.join(path_nested_folders["METRIC_EVAL_FIG"], "{}_vs_z.png".format(metric))
    fig.savefig(filename, dpi=150)
    plt.close(fig)
    print(f"Saved: {filename}")

#### Mapped model-metric-value for each time window of size z

In [ ]:
content_z_value = load_json_content(path_files["EVAL_METRICS"])

mapped_metrics = pd.DataFrame()
for content in content_z_value:
    for metric in EVALUATION_METRICS:
        new_map = {
            "z_value": content["z_value"],
            "model_name": content["model_name"],
            "metric": metric,
            "value": [content[metric]]
        }
        mapped_metrics = pd.concat([mapped_metrics, pd.DataFrame(new_map, index=[0])], ignore_index=True)

for metric in EVALUATION_METRICS:
    selected_metric_per_z = pd.DataFrame()
    for model in CLASSIFIERS:
        new_mapped_values = mapped_metrics[(mapped_metrics["model_name"] == model) & (mapped_metrics["metric"] == metric)]
        selected_metric_per_z = pd.concat([selected_metric_per_z, new_mapped_values],ignore_index=True)
    plot_metrics_vs_z(selected_metric_per_z)